In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 500
num_features = 10

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0,    # X5 effect
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 1

#Y_raw = X @ beta_true + noise  + X[:,4:5]*X[:,4:5] +X[:,2:3]*X[:,3:4]
Y_raw = X @ beta_true + noise  +X[:,2:3]*X[:,3:4]+ X[:,4:5]*X[:,4:5]

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 2.0081],
        [ 1.9869],
        [-1.5884],
        [ 0.5902],
        [-0.1569],
        [ 2.9720],
        [-0.0489],
        [-0.2257],
        [-0.0583],
        [ 0.0520],
        [ 0.0306]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([12, 12])

Attention Matrix:
 tensor([[1.2385e-01, 6.4521e-02, 1.0264e-01, 9.3421e-02, 1.1514e-01, 5.5243e-02,
         1.0407e-01, 1.0592e-01, 7.4527e-02, 7.8705e-02, 7.2058e-02, 9.9078e-03],
        [7.2859e-02, 9.2407e-02, 9.1115e-02, 1.1479e-01, 7.2858e-02, 7.4166e-02,
         1.0024e-01, 5.6876e-02, 7.4202e-02, 6.8333e-02, 5.3670e-02, 1.2849e-01],
        [7.6304e-02, 1.4212e-01, 1.3609e-01, 8.9584e-02, 6.2350e-02, 9.8763e-02,
         1.0385e-01, 5.2032e-02, 8.0234e-02, 6.0751e-02, 4.3927e-02, 5.4002e-02],
        [6.6767e-02, 8.6684e-02, 1.3061e-01, 1.2778e-01, 7.5183e-02, 7.7056e-02,
         1.0147e-01, 7.6697e-02, 6.1088e-02, 7.0185e-02, 1.2074e-01, 5.7393e-03],
        [6.8546e-02, 6.0079e-02, 3.8501e-02, 6.8801e-02, 7.7856e-02, 5.9732e-02,
         7.5052e-02, 4.6027e-02, 5.1721e-02, 6.0215e-02, 7.1048e-02, 3.2242e-01],
        [7.5660e-02, 7.8303e-02, 9.1429e-02, 1.3568e-01, 8.1589e-02, 7.2438e-02,
         7.0464e-02, 9.2689e-02, 1.1314

In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

#A = scores[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([11, 11])

Attention Matrix(拿掉Y):
 tensor([[0.1238, 0.0645, 0.1026, 0.0934, 0.1151, 0.0552, 0.1041, 0.1059, 0.0745,
         0.0787, 0.0721],
        [0.0729, 0.0924, 0.0911, 0.1148, 0.0729, 0.0742, 0.1002, 0.0569, 0.0742,
         0.0683, 0.0537],
        [0.0763, 0.1421, 0.1361, 0.0896, 0.0623, 0.0988, 0.1038, 0.0520, 0.0802,
         0.0608, 0.0439],
        [0.0668, 0.0867, 0.1306, 0.1278, 0.0752, 0.0771, 0.1015, 0.0767, 0.0611,
         0.0702, 0.1207],
        [0.0685, 0.0601, 0.0385, 0.0688, 0.0779, 0.0597, 0.0751, 0.0460, 0.0517,
         0.0602, 0.0710],
        [0.0757, 0.0783, 0.0914, 0.1357, 0.0816, 0.0724, 0.0705, 0.0927, 0.1131,
         0.0754, 0.0855],
        [0.0778, 0.0678, 0.0884, 0.0577, 0.0969, 0.0836, 0.0928, 0.0694, 0.0797,
         0.1165, 0.1034],
        [0.0716, 0.0964, 0.0601, 0.0699, 0.0311, 0.0546, 0.0499, 0.0484, 0.1192,
         0.0685, 0.0629],
        [0.0852, 0.1017, 0.0572, 0.1070, 0.0741, 0.0828, 0.0943, 0.11

In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[2.5996, 1.3543, 2.1544, 1.9609, 2.4169, 1.1596, 2.1844, 2.2233, 1.5643,
         1.6520, 1.5125],
        [1.5293, 1.9396, 1.9125, 2.4095, 1.5293, 1.5568, 2.1040, 1.1938, 1.5575,
         1.4343, 1.1266],
        [1.6016, 2.9830, 2.8566, 1.8804, 1.3087, 2.0731, 2.1797, 1.0922, 1.6841,
         1.2752, 0.9220],
        [1.4015, 1.8195, 2.7415, 2.6821, 1.5781, 1.6174, 2.1300, 1.6099, 1.2823,
         1.4732, 2.5343],
        [1.4388, 1.2611, 0.8081, 1.4441, 1.6342, 1.2538, 1.5754, 0.9661, 1.0856,
         1.2639, 1.4913],
        [1.5881, 1.6436, 1.9191, 2.8481, 1.7126, 1.5205, 1.4791, 1.9456, 2.3749,
         1.5826, 1.7947],
        [1.6331, 1.4233, 1.8555, 1.2119, 2.0332, 1.7544, 1.9480, 1.4572, 1.6722,
         2.4463, 2.1705],
        [1.5036, 2.0237, 1.2620, 1.4682, 0.6527, 1.1454, 1.0474, 1.0161, 2.5019,
         1.4383, 1.3213],
        [1.7890, 2.1347, 1.2007, 2.2466, 1.5553, 1.7380, 1.9793, 2.3368, 1.3469,
         1.5310, 1.7308],
        [1.1678, 2.3029, 2.1704, 1.46

In [11]:
X_train.T @ X_train

tensor([[400.0000,  -0.4857,   9.4604,  29.9864,  -0.9176,  30.8340,  -2.5754,
          18.9926, -20.7359, -10.8697,  26.7517],
        [ -0.4857, 391.9021, -19.2311, -12.4515, -24.1543,  18.7191,  27.2381,
         -40.1990,   8.8554,  -0.8612,   5.7143],
        [  9.4604, -19.2311, 412.0198,   2.3193,  14.8947,  34.3325,  -4.5713,
         -35.4869,  13.9179,  17.5758,   7.2025],
        [ 29.9864, -12.4515,   2.3193, 431.0558,  11.9687,  -4.4587,  26.4212,
          -5.9799,  19.9134,  -8.9648,  13.9600],
        [ -0.9176, -24.1543,  14.8947,  11.9687, 427.9239,  11.5839,  -6.1479,
          21.0333, -22.6914,  10.4535,  -2.8320],
        [ 30.8340,  18.7191,  34.3325,  -4.4587,  11.5839, 424.7664, -21.4397,
          26.0077,  10.0448,   2.7725,  -3.4240],
        [ -2.5754,  27.2381,  -4.5713,  26.4212,  -6.1479, -21.4397, 381.8614,
          10.0332,   2.3798, -14.3425,  21.8681],
        [ 18.9926, -40.1990, -35.4869,  -5.9799,  21.0333,  26.0077,  10.0332,
         468.4420,

In [12]:
beta_attention = (torch.linalg.solve(X_train.T @ X_train + A.T@A, X_train.T @ Y_train)+torch.linalg.solve(X_train.T @ X_train,X_train.T @ Y_train))/2

In [13]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

In [14]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)



OLS beta: tensor([[ 2.0736],
        [ 2.1179],
        [-1.5495],
        [ 0.5347],
        [-0.0734],
        [ 2.9474],
        [-0.0130],
        [-0.2069],
        [-0.0789],
        [ 0.0270],
        [-0.0429]])


In [15]:
print(
    "Attention MSE:",
    mse_attention.item()
)

print()

print(
    "OLS MSE:      ",
    mse_ols.item()
)

Attention MSE: 4.1535491943359375

OLS MSE:       4.154486179351807
